# Rebuild `sensory_postsynaptic_partners.csv` (input to Figure 3e)

Figure 3e counts, for each sensory neuron, how many **distinct, intact** spinal cord neurons
it synapses onto. That needs two things per postsynaptic partner: the number of synapses it
receives, and a measure of how completely it is reconstructed. This notebook builds that
table.

For every one of the 79 reconstructed sensory neurons:

1. Query CAVE `synapses_version3` with the sensory neuron **presynaptic**.
2. Drop autapses and synapses with cleft `size` < 20 voxels — identical filtering to
   Figure 3d, so the two panels describe the same set of synapses.
3. Group by `post_pt_root_id` → **`num_synapses`**.
4. For each distinct partner, **`num_supervoxels` = `len(chunkedgraph.get_leaves(root_id))`**
   — the number of level-0 supervoxels agglomerated into that object. Small values mean an
   unproofread fragment; large values mean a substantially reconstructed neuron.

Output columns: `pre_pt_root_id, post_pt_root_id, num_supervoxels, num_synapses`.

## Provenance note

The original `final.csv` was produced in February 2026 by a script that was not kept. The
recipe above was **reverse-engineered from the file itself and verified against live CAVE**:
`len(get_leaves(...))` reproduced the stored `num_supervoxels` exactly for 8/8 spot-checked
partners, and `synapses_version3` with `size >= 20` reproduced the stored `num_synapses` for
100% of the 864 overlapping partners of a test sensory neuron. Remaining differences are
proofreading drift since February.

## Runtime

Step 4 is one ChunkedGraph call per distinct partner. There are ~33,000 distinct partners
and the call runs at roughly 6/s, so a full rebuild takes about 1.5 hours. The counts are
cached to `supervoxel_count_cache.json` and the notebook is resumable — re-running skips
everything already cached, so an interrupted run can simply be restarted.


In [ ]:
import json
from pathlib import Path

import caveclient
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Pin the materialization so a rebuild is reproducible.
MATERIALIZATION = 669

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")
if MATERIALIZATION is not None:
    client.version = MATERIALIZATION

SYNAPSE_TABLE = "synapses_version3"
MIN_CLEFT_SIZE = 20                            # voxels, matches Figure 3d
CACHE = Path("supervoxel_count_cache.json")    # resumable
OUT_CSV = Path("../../data/sensory_postsynaptic_partners.csv")


In [ ]:
# The 79 reconstructed sensory neurons, grouped by subtype
# (10 per subtype except C-Cold, n = 9).
SENSORY_NEURONS = {
    "c-ltmr": [
        720575940886931959, 720575940891163574, 720575940922958792,
        720575940918770481, 720575940893408011, 720575940880372359,
        720575940918990151, 720575940902671774, 720575940893122730,
        720575940914869232,
    ],
    "mrgd": [
        720575940915684544, 720575940921986112, 720575940920833705,
        720575940915967584, 720575940901960430, 720575940864403212,
        720575940915217983, 720575940927208416, 720575940941089388,
        720575940899256770,
    ],
    "ad-ltmr": [
        720575940902363320, 720575940879973250, 720575940909767128,
        720575940893392149, 720575940909235164, 720575940887704904,
        720575940909767384, 720575940913511793, 720575940908252144,
        720575940914044656,
    ],
    "ab-ltmr": [
        720575940882531294, 720575940893873415, 720575940881926778,
        720575940932756628, 720575940905014742, 720575940871491897,
        720575940892205986, 720575940884949941, 720575940898835143,
        720575940899788943,
    ],
    "sst": [
        720575940882155492, 720575940905081398, 720575940883866908,
        720575940918775345, 720575940882061434, 720575940920062581,
        720575940891751079, 720575940884108031, 720575940911489471,
        720575940927216096,
    ],
    "ad-htmr": [
        720575940903122154, 720575940884031660, 720575940884051238,
        720575940897347842, 720575940847552365, 720575940917989973,
        720575940922765354, 720575940882082080, 720575940884977867,
        720575940902680734,
    ],
    "chtmr-p": [
        720575940918288693, 720575940905365705, 720575940862564127,
        720575940916233731, 720575940881918801, 720575940910952353,
        720575940924479298, 720575940874972385, 720575940892182782,
        720575940914066622,
    ],
    "c-cold": [
        720575940872856946, 720575940883734520, 720575940890741967,
        720575940899184211, 720575940902618322, 720575940927175960,
        720575940927275084, 720575940878947380, 720575940900244450,
    ],
}


In [ ]:
def resolve_current_ids(client, root_ids):
    """Replace root IDs made stale by later proofreading with their successors."""
    ids = np.asarray([int(r) for r in root_ids], dtype=np.uint64)
    mapping, changed = {}, 0
    for rid, ok in zip(root_ids, client.chunkedgraph.is_latest_roots(ids)):
        if ok:
            mapping[rid] = rid
            continue
        mapping[rid] = int(np.atleast_1d(
            client.chunkedgraph.suggest_latest_roots(int(rid)))[0])
        changed += 1
        print(f"  stale: {rid} -> {mapping[rid]}")
    print(f"{changed} of {len(root_ids)} published root IDs were updated.")
    return mapping


current = resolve_current_ids(
    client, [r for ids in SENSORY_NEURONS.values() for r in ids]
)


In [ ]:
# ---- 1-3: synapses per (sensory neuron, postsynaptic partner) pair
pairs = []
for subtype, root_ids in tqdm(list(SENSORY_NEURONS.items()), desc="sensory subtypes"):
    for root_id in root_ids:
        syn = client.materialize.query_table(
            SYNAPSE_TABLE,
            filter_equal_dict={"pre_pt_root_id": current[root_id]},
        )
        if syn is None or syn.empty:
            continue
        syn = syn[
            (syn["pre_pt_root_id"] != syn["post_pt_root_id"])
            & (syn["size"] >= MIN_CLEFT_SIZE)
        ]
        counts = syn.groupby("post_pt_root_id").size()
        for post_id, n in counts.items():
            # report under the PUBLISHED id so the table joins to the figure notebooks
            pairs.append({"pre_pt_root_id": root_id,
                          "post_pt_root_id": int(post_id),
                          "num_synapses": int(n)})

pairs = pd.DataFrame(pairs)
print(f"{len(pairs):,} pre/post pairs, "
      f"{pairs.post_pt_root_id.nunique():,} distinct partners")


In [ ]:
# ---- 4: supervoxel count per distinct partner (slow; cached and resumable)
cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
todo = [int(r) for r in pairs.post_pt_root_id.unique() if str(r) not in cache]
print(f"{len(cache):,} cached, {len(todo):,} to fetch")

for i, rid in enumerate(tqdm(todo, desc="supervoxel counts")):
    try:
        cache[str(rid)] = int(len(client.chunkedgraph.get_leaves(rid)))
    except Exception as exc:                     # keep going; retry on a later run
        print(f"  {rid}: {type(exc).__name__}")
    if i % 500 == 0:
        CACHE.write_text(json.dumps(cache))      # checkpoint

CACHE.write_text(json.dumps(cache))
print(f"{len(cache):,} supervoxel counts cached")


In [ ]:
# ---- assemble and write
pairs["num_supervoxels"] = pairs.post_pt_root_id.astype(str).map(cache)
out = pairs.dropna(subset=["num_supervoxels"]).copy()
out["num_supervoxels"] = out["num_supervoxels"].astype(int)
out = out[["pre_pt_root_id", "post_pt_root_id", "num_supervoxels", "num_synapses"]]

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}  ({len(out):,} rows)")

# Sanity check: the intact filter used by Figure 3e should retain ~9% of partners.
intact = out[out.num_supervoxels >= 40_000]
print(f"partners with >=40,000 supervoxels: "
      f"{intact.post_pt_root_id.nunique():,} / {out.post_pt_root_id.nunique():,}")
